In [1]:
import json
# Sample JSON file
json_file = "../data-out/gems3/from_tmatch/2/DComp.exportGems3"

SDref = "dSDref"#"dSDref"
SDval = "dSDval"#"dSDval"
psi2020 = "2023HUM/THO" # "2024MIR"
tdb2020 = "2024MIR"
# change bellow to
                    #else:
                     #   dod[18]["val"][0] = "Miron:2025:rep:"
                     #   dod[19]["val"][0] = "PSINa25"
def read_json(file_path):
    with open(file_path, "r", encoding="utf-8") as file:
        data = json.load(file)
    return data

def process_json(data):
    for item in data:
        print(f"Processing item with key: {item.get('key', [])}")
        for entry in item.get("dod", []):
            print(f"ID: {entry.get('id', 'N/A')}, Label: {entry.get('label', 'Unknown')}, Value: {entry.get('val', 'None')}")

def save_json(data, filename="formatted_references.json"):
    """Save formatted data to a JSON file."""
    with open(filename, "w", encoding="utf-8") as f:
        json.dump(data, f, indent=4)

In [2]:
# Read and process JSON data
json_data = read_json(json_file+ ".json")
#process_json(json_data)
# Save formatted data to a JSON file
save_json(json_data, json_file+ ".json")

In [3]:
import pandas as pd
def load_references(file_path):
    """Load reference data from Excel file."""
    df = pd.read_excel(file_path, sheet_name="Sheet1")
    ref_dict = dict(zip(df["Ref_abb"].str.replace(" ", ""), df["Ref_GEMS"]))  # Remove spaces for comparison
    return ref_dict

In [4]:
# Load reference data
reference_file = "../data-out/gems3/from_tmatch/references.xlsx"
reference_data = load_references(reference_file)

In [5]:
SD_COUNT = {"dSDref": ("DC_cnt", 2), "rSDref": ("RE_dim", 7)}   # where GEMS keeps the number of references


def set_sd_count(dod):
    """Set the record's reference count to the length of its reference list (DComp: DC_cnt[0][2], ReacDC: RE_dim[0][7])."""
    field, index = SD_COUNT[SDref]
    if isinstance(dod.get(field), list) and dod[field] and len(dod[field][0]) > index:
        dod[field][0][index] = len(dod[SDref]) if isinstance(dod[SDref], list) else 0


def process_items_dSDref_dSDval(data, references):
    """Process item dSDref and update item dSDval based on reference mapping, keeping the two lists paired."""
    for item in data:
        dod = item.get("dod", [])
        dSDref = dod[SDref]
        dSDval = dod[SDval]
        if not isinstance(dSDref, list):
            continue
        pairs = []   # (reference, value)
        if isinstance(dSDval, list):
            for index, value in enumerate(dSDref):
                if ":" in value[0]:
                    # Case 1: If ':' exists, split and process both parts
                    first_part, second_part = value[0].split(":", 1)
                    second_part = second_part.replace(" ", "")  # Remove spaces
                    mapped = references.get(second_part, second_part)  # Replace if match exists
                    pairs.append((mapped if mapped != second_part else f"{value}", first_part))
                else:
                    # Case 2: If no ':', check if value matches Ref_abb
                    new_value = references.get(value[0], value[0])  # Replace if match exists
                    pairs.append((new_value, dSDval[index][0] if index < len(dSDval) and dSDval[index] else ""))
        else:
            for index, value in enumerate(dSDref):
                if ":" not in value[0]:
                    # Case 2: If no ':', check if value matches Ref_abb
                    cleaned_value = value[0].replace(" ", "")  # Remove spaces for matching
                    pairs.append((references.get(cleaned_value, value[0]), value[0]))
                elif "Vm_Ref:" in value[0]:
                    pairs += [("Hummel_ea:2023:dat:", "TDB2020"), ("Miron:2024:rep:", "Vm_Ref")]
        pairs = [(r, v) for r, v in pairs if r != ""]   # a reference is dropped together with its value
        dod[SDref] = [[r] for r, v in pairs]
        dod[SDval] = [[v] for r, v in pairs]
        set_sd_count(dod)
    return data

In [6]:
processed_data = process_items_dSDref_dSDval(json_data, reference_data)

In [7]:
#processed_data

def update_vm_ref_values(data):
    """Replace a "Vm_Ref:..." reference by the pairs Hummel_ea:2023 -> TDB2020 and Miron:2024 -> Vm_Ref."""
    for item in data:
        dod = item.get("dod", [])
        dSDref, dSDval = dod[SDref], dod[SDval]
        if not isinstance(dSDref, list):
            continue
        refs, vals = [], []
        for index, value in enumerate(dSDref):
            if "Vm_Ref:" in value[0]:
                refs += [["Hummel_ea:2023:dat:"], ["Miron:2024:rep:"]]
                vals += [["TDB2020"], ["Vm_Ref"]]
            else:
                refs.append(value)
                vals.append(dSDval[index] if isinstance(dSDval, list) and index < len(dSDval) else [""])
        dod[SDref], dod[SDval] = refs, vals
        set_sd_count(dod)
    return data

processed_data = update_vm_ref_values(processed_data)

In [8]:
#processed_data

In [9]:
# Save formatted data to a JSON file
save_json(processed_data, json_file+ "_formatted"+ ".json")